# Semantic Reasoning Dispersion Notebook

This notebook implements a ready-to-run pipeline to compute **semantic dispersion per question** using **step-level embeddings + step-alignment (RSS practical variant)** and a fallback **full-output embedding** dispersion. It is model-agnostic and can use either **local sentence-transformers** models (open-source) or **OpenAI embeddings** if you prefer.

### What it computes
- `avg_pairwise_step_similarity`: average matched-step similarity across models for each question
- `dispersion_steps = 1 - avg_pairwise_step_similarity`
- `avg_pairwise_full_similarity`: average pairwise similarity of full outputs
- `dispersion_full = 1 - avg_pairwise_full_similarity`

### How to run
1. Upload your CSV dataset (columns: `model,question_id,question,A,B,C,D,correct_answer,model_answer,is_correct,causal_reasoning,raw_output`).
2. Choose `backend = 'local'` (sentence-transformers) or `backend = 'openai'` (requires `OPENAI_API_KEY`).
3. Run the `compute_dispersion(...)` cell.

Notes: this notebook creates an output CSV with dispersion metrics per question.


## 1) Requirements

Install the required packages **locally** before running (recommended):
```bash
pip install pandas numpy scipy scikit-learn sentence-transformers openai
```
- If you *cannot* install packages, use the `backend='openai'` option and set `OPENAI_API_KEY` in your environment.
- The notebook *does not* run package installs automatically here; run the command above in your environment.


In [ ]:
import ast
import json
from itertools import combinations
from typing import List

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment
from sklearn.metrics.pairwise import cosine_similarity

# Optional imports: sentence-transformers or openai. If not available, set backend appropriately.
try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

try:
    import openai
except Exception:
    openai = None

def parse_causal_field(s):
    """Parse causal_reasoning that may be JSON string, Python list string, or already a list."""
    if pd.isna(s) or s is None or s == "":
        return []
    if isinstance(s, list):
        return s
    # Try JSON
    try:
        parsed = json.loads(s)
        if isinstance(parsed, dict) and "causal_reasoning" in parsed:
            return parsed["causal_reasoning"]
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    # Try ast literal eval
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    # Fallback: split lines
    if isinstance(s, str):
        lines = [ln.strip() for ln in s.splitlines() if ln.strip()]
        if len(lines) > 1:
            return lines
        return [s.strip()]
    return []

class EmbeddingBackend:
    def __init__(self, backend="auto", openai_model="text-embedding-3-large", local_model_name="all-MiniLM-L6-v2"):
        self.backend = backend
        self.openai_model = openai_model
        self.local_model_name = local_model_name
        self.local_model = None
        if backend == "auto":
            if openai is not None and 'OPENAI_API_KEY' in globals() and globals().get('OPENAI_API_KEY'):
                self.backend = 'openai'
            elif SentenceTransformer is not None:
                self.backend = 'local'
            else:
                # prefer local if available
                self.backend = 'local' if SentenceTransformer is not None else 'openai'
        if self.backend == "local":
            if SentenceTransformer is None:
                raise RuntimeError("sentence-transformers not available. Install it or use OpenAI backend.")
            self.local_model = SentenceTransformer(local_model_name)
        if self.backend == "openai":
            if openai is None:
                raise RuntimeError("openai package not available. Install it or use local backend.")
            if 'OPENAI_API_KEY' not in (env := globals()):
                # try environment variable
                import os
                if 'OPENAI_API_KEY' not in os.environ:
                    raise RuntimeError("OPENAI_API_KEY env var not set. Set it or use local backend.")
                openai.api_key = os.environ['OPENAI_API_KEY']
            else:
                openai.api_key = globals()['OPENAI_API_KEY']

    def embed(self, texts: List[str]) -> np.ndarray:
        if len(texts) == 0:
            return np.zeros((0, 1))
        if self.backend == 'local':
            embs = self.local_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
            return embs
        elif self.backend == 'openai':
            embeds = []
            chunk = []
            for t in texts:
                chunk.append(t)
                if len(chunk) >= 256:
                    resp = openai.Embedding.create(model=self.openai_model, input=chunk)
                    for r in resp['data']:
                        embeds.append(r['embedding'])
                    chunk = []
            if chunk:
                resp = openai.Embedding.create(model=self.openai_model, input=chunk)
                for r in resp['data']:
                    embeds.append(r['embedding'])
            return np.array(embeds, dtype=float)
        else:
            raise RuntimeError(f"Unknown backend {self.backend}")

def pad_similarity_matrix(sim, n):
    m, k = sim.shape
    if m == n and k == n:
        return sim
    mat = np.zeros((n, n))
    mat[:m, :k] = sim
    return mat

def matched_step_similarity(steps_a: List[str], steps_b: List[str], embedder: EmbeddingBackend) -> float:
    if len(steps_a) == 0 and len(steps_b) == 0:
        return 1.0
    if len(steps_a) == 0 or len(steps_b) == 0:
        return 0.0
    emb_a = embedder.embed(steps_a)
    emb_b = embedder.embed(steps_b)
    sim = cosine_similarity(emb_a, emb_b)
    n = max(sim.shape)
    sim_padded = pad_similarity_matrix(sim, n)
    cost = 1.0 - sim_padded
    row_ind, col_ind = linear_sum_assignment(cost)
    matched_sims = sim_padded[row_ind, col_ind]
    mean_sim = matched_sims.mean()
    return float(mean_sim)

def pairwise_step_dispersion(step_lists: List[List[str]], embedder: EmbeddingBackend):
    sims = []
    for a, b in combinations(step_lists, 2):
        sim = matched_step_similarity(a, b, embedder)
        sims.append(sim)
    if len(sims) == 0:
        return {"avg_pairwise_step_similarity": 1.0, "dispersion_steps": 0.0}
    avg_sim = float(np.mean(sims))
    return {"avg_pairwise_step_similarity": avg_sim, "dispersion_steps": 1.0 - avg_sim}

def pairwise_full_output_dispersion(full_texts: List[str], embedder: EmbeddingBackend):
    if len(full_texts) <= 1:
        return {"avg_pairwise_full_similarity": 1.0, "dispersion_full": 0.0}
    embs = embedder.embed(full_texts)
    sims = []
    for i, j in combinations(range(len(full_texts)), 2):
        sims.append(float(cosine_similarity(embs[i : i + 1], embs[j : j + 1])[0, 0]))
    avg_sim = float(np.mean(sims))
    return {"avg_pairwise_full_similarity": avg_sim, "dispersion_full": 1.0 - avg_sim}

def compute_dispersion(df: pd.DataFrame, backend='local', openai_model='text-embedding-3-large', local_model='all-MiniLM-L6-v2'):
    embedder = EmbeddingBackend(backend=backend, openai_model=openai_model, local_model_name=local_model)
    results = []
    if 'causal_reasoning' not in df.columns:
        raise ValueError("DataFrame must contain 'causal_reasoning' column")
    for qid, group in df.groupby('question_id'):
        step_lists = []
        raw_texts = []
        models = list(group['model'].unique())
        for _, row in group.iterrows():
            steps = parse_causal_field(row['causal_reasoning'])
            steps_norm = []
            for s in steps:
                s = s.strip()
                if len(s) > 2 and s[0].isdigit() and s[1] in ['.', ')', '-']:
                    s = s[2:].strip()
                steps_norm.append(s)
            step_lists.append(steps_norm)
            ro = row.get('raw_output', '')
            if isinstance(ro, str) and ro.strip():
                raw_texts.append(ro)
            else:
                raw_texts.append(' '.join(steps_norm))
        step_metrics = pairwise_step_dispersion(step_lists, embedder)
        full_metrics = pairwise_full_output_dispersion(raw_texts, embedder)
        results.append({
            'question_id': qid,
            'num_models': len(group),
            'models': ';'.join(sorted(models)),
            **step_metrics,
            **full_metrics,
        })
    return pd.DataFrame(results)

print('Utility functions loaded. Use compute_dispersion(df, backend="local") to compute metrics.')


## 2) Example usage
Replace `data.csv` with your dataset path. The CSV must include a `causal_reasoning` column that is either:
- a JSON dict with key `causal_reasoning` containing a list of steps, or
- a Python-list-like string (e.g., `["1. ...","2. ..."]`), or
- a multi-line string with each step on a new line.


In [ ]:
# Example: load CSV and compute dispersion
input_path = 'data.csv'  # change to your file
df = pd.read_csv(input_path, dtype=str).fillna('')
print('Loaded rows:', len(df))
# Choose backend: 'local' for sentence-transformers, 'openai' for OpenAI embeddings
backend = 'local'
out_df = compute_dispersion(df, backend=backend)
out_df.to_csv('dispersion_by_question.csv', index=False)
print('Wrote dispersion_by_question.csv')
out_df.head()


## 3) Next steps / Extensions
- Export per-pair similarities: modify `compute_dispersion` to return per-pair values.
- Visualize dispersion distribution with histograms or UMAP.
- Combine dispersion with cross-model perplexity asymmetry for MAD selection.
- Add step-level factual grounding checks using retrieval or DoWhy/EconML mappings.
